# Halifax Digital Twin - Sionna RT Demo

This notebook validates the Halifax digital twin scene, places cellular transmitters, and runs radio-propagation simulations with Sionna RT.

### Table of Contents
1. [Environment and Project Setup](#setup)
2. [Quick Visual Checks](#visual-checks)
3. [Radio Material Model](#materials)
4. [Load Sionna Scene](#scene)
5. [Select and Validate Transmitters](#tx-selection)
6. [Simple Radio Map](#simple-radiomap)
7. [Optional Advanced Simulations](#advanced)
8. [Radio Metrics](#metrics)
9. [3D Preview and Ray Paths](#ray-paths)



## 1. Environment and Project Setup <a id='setup'></a>

Configure the execution backend, import dependencies, resolve project paths, and load the antenna dataset.


In [ ]:
import os

# Runtime backend. Change USE_GPU, then restart the kernel and run all cells.
# The backend must be selected before importing Mitsuba, Sionna, Dr.Jit, or Torch.
USE_GPU = False  # True = GPU/CUDA, False = CPU/LLVM

if USE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["MITSUBA_VARIANT"] = "cuda_ad_mono_polarized"
    print("Runtime mode: GPU")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    os.environ["MITSUBA_VARIANT"] = "llvm_ad_mono_polarized"
    print("Runtime mode: CPU")

print("CUDA_VISIBLE_DEVICES =", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("MITSUBA_VARIANT =", os.environ.get("MITSUBA_VARIANT"))

import mitsuba as mi

_mitsuba_variant = os.environ["MITSUBA_VARIANT"]
try:
    mi.set_variant(_mitsuba_variant)
except Exception as exc:
    active_variant = None
    try:
        active_variant = mi.variant()
    except Exception:
        pass
    if active_variant != _mitsuba_variant:
        raise RuntimeError(
            f"Could not initialize Mitsuba variant '{_mitsuba_variant}'.\n"
            "Recommended on Jetson/NVIDIA GPU: cuda_ad_mono_polarized.\n"
            "CPU fallback: llvm_ad_mono_polarized. Windows CPU mode also needs LLVM."
        ) from exc

print(f"Mitsuba variant: {mi.variant()}")

import sys, json, math, warnings, subprocess
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from pyproj import Transformer
from pathlib import Path

import gc, re, time
print(f"Sionna/Mitsuba backend: {mi.variant()}")

import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection
from matplotlib.ticker import FuncFormatter
from matplotlib.colors import BoundaryNorm, ListedColormap

import pyproj
from shapely.geometry import shape, Point
from shapely.affinity import translate
from shapely.validation import make_valid
from shapely.ops import transform as shp_transform
from scipy.interpolate import make_smoothing_spline, PchipInterpolator
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.crs import CRS

import folium
from folium.plugins import MarkerCluster, HeatMap

import sionna
from sionna.rt import (load_scene, Transmitter, Receiver, PlanarArray,
                        RadioMaterial, RadioMapSolver, PathSolver)

_rm_solver   = RadioMapSolver()  
_path_solver = PathSolver()      

warnings.filterwarnings("ignore")



def get_tx_power_dbm(z_m: float) -> float:
    if z_m >= 20: return 43.0
    if z_m >= 10: return 38.0
    return 24.0

# Project paths and current Halifax inputs

def find_project_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root containing data")

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"
DATA_DIR = DATA_ROOT / "processed_data"
SCENE_DIR = DATA_ROOT / "scenes" / "halifax_peninsula"
OUT_DIR = PROJECT_ROOT / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENE_XML_MULTI = SCENE_DIR / "halifax_peninsula_multi_material.xml"
SCENE_XML_MONO_CONCRETE = SCENE_DIR / "halifax_peninsula_mono_concrete.xml"
SCENE_XML = str(SCENE_XML_MULTI)
ANTENNAS_CSV = DATA_DIR / "peninsula_cellular_antennas_2g_to_5g.csv"
BUILDING_HEIGHTS_GPKG = DATA_DIR / "building_heights_selected.gpkg"
OUT = str(OUT_DIR)

# Default simulation frequency used for the first scene load and radio-map test.
# Change this value before running the notebook if you want another band.
FREQUENCY = 3500e6
FREQUENCY_WINDOW_MHZ = 200.0

PENINSULA_COORDS = [
    (-63.6194204, 44.6410835),
    (-63.6308541, 44.6643012),
    (-63.6220173, 44.6817374),
    (-63.5992474, 44.6736948),
    (-63.5531831, 44.6413834),
    (-63.5565099, 44.6169733),
    (-63.5641958, 44.6133803),
    (-63.5844221, 44.6253175),
    (-63.6194298, 44.6410626),
    (-63.6194204, 44.6410835),
]
PENINSULA_FEATURE_COLLECTION = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {},
        "geometry": {"type": "Polygon", "coordinates": [PENINSULA_COORDS]},
    }],
}


def read_terrain_metadata(path):
    metadata = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if ":" not in line:
                continue
            key, value = line.strip().split(":", 1)
            value = value.strip()
            try:
                metadata[key] = float(value)
            except ValueError:
                metadata[key] = value
    return metadata

terrain_meta = read_terrain_metadata(SCENE_DIR / "terrain_metadata.txt")
X_ORI = float(terrain_meta["local_origin_x_utm_m"])
Y_ORI = float(terrain_meta["local_origin_y_utm_m"])
X_MAX = float(terrain_meta["local_max_x_m"])
Y_MAX = float(terrain_meta["local_max_y_m"])
CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

def load_antenna_dataset(path):
    raw = pd.read_csv(path)
    tr = Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
    lon = pd.to_numeric(raw["LONGITUDE"], errors="coerce")
    lat = pd.to_numeric(raw["LATITUDE"], errors="coerce")
    x_utm, y_utm = tr.transform(lon.to_numpy(), lat.to_numpy())

    ground = pd.to_numeric(raw.get("DEM_GROUND_ELEV_M"), errors="coerce")
    ant_z = pd.to_numeric(raw.get("ANTENNA_Z_M"), errors="coerce")
    ant_height = pd.to_numeric(raw.get("TX_ANT_HT"), errors="coerce")
    z = ant_z.fillna(ground + ant_height).fillna(30.0)

    df = pd.DataFrame({
        "lat": lat,
        "lon": lon,
        "x_local_m": x_utm - X_ORI,
        "y_local_m": y_utm - Y_ORI,
        "z_local_m": z,
        "freq_mhz": pd.to_numeric(raw["TRANSMIT_FREQ"], errors="coerce"),
        "bw_mhz": pd.to_numeric(raw.get("TRANSMIT_BW"), errors="coerce").fillna(0.0),
        "azimuth_deg": pd.to_numeric(raw.get("TX_ANT_AZIM"), errors="coerce").fillna(0.0),
        "tilt_deg": pd.to_numeric(raw.get("TX_ANT_ELEV_ANGLE"), errors="coerce").fillna(0.0),
        "gain_dbi": pd.to_numeric(raw.get("TX_ANT_GAIN"), errors="coerce").fillna(0.0),
        "site_elev_m": pd.to_numeric(raw.get("SITE_ELEV"), errors="coerce").fillna(0.0),
        "mast_ht_m": pd.to_numeric(raw.get("STUCT_HT"), errors="coerce").fillna(0.0),
        "antenna_ht_m": ant_height.fillna(0.0),
        "power_dbm": pd.to_numeric(raw.get("TX_PWR"), errors="coerce"),
        "licensee": raw.get("LICENSEE", "").fillna(""),
        "service": raw.get("SERVICE", "").fillna(""),
        "technology": raw.get("TECHNOLOGY", "").fillna(""),
        "location": raw.get("LOCATION", "").fillna(""),
    })
    df = df.dropna(subset=["lat", "lon", "x_local_m", "y_local_m", "z_local_m", "freq_mhz"]).copy()
    df = df[(df["x_local_m"] >= 0) & (df["x_local_m"] <= X_MAX) &
            (df["y_local_m"] >= 0) & (df["y_local_m"] <= Y_MAX)].reset_index(drop=True)
    return df

df = load_antenna_dataset(ANTENNAS_CSV)

def get_peninsula_shape_local():
    pen = shape(PENINSULA_FEATURE_COLLECTION["features"][0]["geometry"])
    tr = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)
    pen = shp_transform(lambda x, y: tr.transform(x, y), pen)
    return shp_transform(lambda x, y: (x - X_ORI, y - Y_ORI), pen)

for name, file_path in [("SCENE_XML", SCENE_XML), ("ANTENNAS_CSV", ANTENNAS_CSV)]:
    ok = os.path.exists(file_path)
    print(f"  {'OK' if ok else 'X '} {name:20s} = {file_path}")
print(f"\nScene extent : {CM_SIZE_X:.0f} m (E) x {CM_SIZE_Y:.0f} m (N)")
print(f"Antennas     : {len(df)} sectors")

RES_M = 10.0

OP_COLORS = {
    "Bell": "#0066CC",
    "Rogers": "#DA291C",
    "Bragg": "#00A94F",
    "FIDO": "#DA291C",
}

def operator_group(name):
    name_upper = str(name).upper()
    if "FIDO" in name_upper or "ROGERS" in name_upper:
        return "Rogers"
    if "BELL" in name_upper:
        return "Bell"
    if "BRAGG" in name_upper or "EASTLINK" in name_upper:
        return "Bragg"
    if "TELUS" in name_upper:
        return "TELUS"
    return "Other"

def op_color(name):
    return OP_COLORS.get(operator_group(name), OP_COLORS.get(str(name), "#888888"))

print("Setup OK")








## 2. Quick Visual Checks <a id='visual-checks'></a>

Before running Sionna, verify that the footprints, peninsula boundary, and antennas are spatially consistent.

### 2.1 Peninsula, Footprints, and Antennas


In [ ]:
tr_inv = Transformer.from_crs("EPSG:32620", "EPSG:4326", always_xy=True)

def utm_x_to_lon(x, pos):
    lon, _ = tr_inv.transform(x, 4944000)  
    return f"{lon:.3f}°"

def utm_y_to_lat(y, pos):
    _, lat = tr_inv.transform(452000, y)    
    return f"{lat:.3f}°"

peninsula = PENINSULA_FEATURE_COLLECTION
buildings = json.loads(gpd.read_file(BUILDING_HEIGHTS_GPKG).to_crs("EPSG:4326").to_json())

tr = Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)

def project_ring(coords):
    xs, ys = tr.transform([c[0] for c in coords], [c[1] for c in coords])
    return list(zip(xs, ys))

fig, ax = plt.subplots(figsize=(11, 13))

# Buildings 
patches = []
for feat in buildings["features"]:
    geom = feat["geometry"]
    rings = []
    if geom["type"] == "Polygon":
        rings = [geom["coordinates"][0]]
    elif geom["type"] == "MultiPolygon":
        rings = [poly[0] for poly in geom["coordinates"]]
    for ring in rings:
        pts = project_ring(ring)
        patches.append(MplPolygon(pts, closed=True))

ax.add_collection(PatchCollection(
    patches, facecolor="#C8B89A", edgecolor="#B0A080",
    linewidth=0.15, alpha=0.85, zorder=2
))

# Peninsula boundary 
pen_coords = peninsula["features"][0]["geometry"]["coordinates"][0]
pen_pts    = project_ring(pen_coords)


# Antennas coloured by operator
OP_MAP = {
    "Bell"     : ("#E8372A", "Bell"),
    "Rogers"   : ("#2BBFAF", "Rogers"),
    "TELUS"    : ("#6B3FA0", "TELUS"),
    "Bragg"    : ("#F59B1E", "Bragg"),
    
}

def get_op_key(name):
    for k in OP_MAP:
        if k.lower() in str(name).lower():
            return k
    return "Other"


tr = Transformer.from_crs("EPSG:4326", "EPSG:32620", always_xy=True)

plotted = set()

for _, row in df.iterrows():
    x_utm, y_utm = tr.transform(row["lon"], row["lat"])
    op  = get_op_key(row["licensee"])
    col = OP_MAP.get(op, ("#888888", "Other"))[0]
    ax.scatter(x_utm, y_utm, c=col, marker="^", s=55,
               zorder=5, linewidths=0.4, edgecolors="white")
    
    plotted.add(op)

legend_handles = [
    mpatches.Patch(color=OP_MAP[k][0], label=OP_MAP[k][1])
    for k in OP_MAP if k in plotted
]
if "Other" in plotted:
    legend_handles.append(mpatches.Patch(color="#888888", label="Other"))

ax.legend(handles=legend_handles, title="Operator",
          loc="upper right", fontsize=9, title_fontsize=9,
          framealpha=0.9)

# Axes & formatting 
pen_xs = [p[0] for p in pen_pts]
pen_ys = [p[1] for p in pen_pts]
margin = 300
ax.set_xlim(min(pen_xs) - margin, max(pen_xs) + margin)
ax.set_ylim(min(pen_ys) - margin, max(pen_ys) + margin)
ax.set_aspect("equal")
ax.xaxis.set_major_formatter(FuncFormatter(utm_x_to_lon))
ax.yaxis.set_major_formatter(FuncFormatter(utm_y_to_lat))
ax.set_xlabel("Longitude", fontsize=11)
ax.set_ylabel("Latitude", fontsize=11)
ax.set_title("Halifax Peninsula", fontsize=13, fontweight="bold")
ax.tick_params(labelsize=9)
ax.grid(False)

plt.tight_layout()
out_path = os.path.join(OUT, "demo_map_halifax.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")




### 2.2 Interactive Antenna Map <a id='antenna-map'></a>

Interactive Folium map of the cellular sectors on the Halifax peninsula.


In [ ]:
LAT_C, LON_C = 44.647, -63.587

m = folium.Map(location=[LAT_C, LON_C], zoom_start=13, tiles="CartoDB positron")

band35 = df[df.freq_mhz.between(3400, 3600)].copy()

for op, grp in band35.groupby("licensee"):
    cluster = MarkerCluster(name=op.split()[0]).add_to(m)
    col = op_color(op)
    for _, r in grp.iterrows():
        popup_html = (
            f"<b>{op}</b><br>"
            f"Freq : {r.freq_mhz:.0f} MHz | BW : {r.bw_mhz:.0f} MHz<br>"
            f"Az   : {r.azimuth_deg:.0f}° | Tilt : {r.tilt_deg:.1f}°<br>"
            f"Gain : {r.gain_dbi:.1f} dBi | Z : {r.z_local_m:.0f} m<br>"
            f"Site: {r.site_elev_m:.0f} m + mast {r.mast_ht_m:.0f} m"
        )
        folium.CircleMarker(
            location=[r.lat, r.lon], radius=6,
            color=col, fill=True, fill_color=col, fill_opacity=0.85,
            tooltip=popup_html,
        ).add_to(cluster)

legend = '''<div style='position:fixed;bottom:30px;left:30px;z-index:9999;
background:white;padding:10px 15px;border-radius:8px;
box-shadow:0 2px 8px rgba(0,0,0,.3);font-family:sans-serif;font-size:12px'>
<b>Antennes</b><br>
<span style='color:#0066CC'>&#9679;</span> Bell<br>
<span style='color:#DA291C'>&#9679;</span> Rogers<br>
<span style='color:#00A94F'>&#9679;</span> Bragg/Eastlink</div>'''
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl().add_to(m)

html_path = os.path.join(OUT, "map_antennas.html")
m.save(html_path)
print(f"Map saved: {html_path}")
display(m)




### 2.3 Antenna Density Heatmap

Builds a heatmap of all antenna sectors across all bands and operators.


In [ ]:
#  Heatmap all antennas 
m2 = folium.Map(location=[LAT_C, LON_C], zoom_start=13, tiles="CartoDB dark_matter")

HeatMap(
    [[r.lat, r.lon, 1.0] for _, r in df.iterrows()],
    radius=14, blur=10, min_opacity=0.35,
    gradient={0.3: "#2C1654", 0.55: "#0D47A1", 0.75: "#00BCD4", 1.0: "#B2FF59"}
).add_to(m2)

folium.map.Marker(
    [44.670, -63.550],
    icon=folium.DivIcon(html=(
        "<div style='font:bold 12px monospace;color:white;white-space:nowrap'>"
        "Halifax — 3 625 secteurs </div>"
    ))
).add_to(m2)

html2 = os.path.join(OUT, "map_heatmap_all.html")
m2.save(html2)
print(f"Heatmap saved: {html2}")
display(m2)




### 2.4 Spectrum Usage by Operator

Grouped bar chart showing how the operators present in the filtered Halifax dataset deploy their spectrum.


In [ ]:
BANDS = [600, 700, 800, 900, 1900, 2100, 2600, 3500]
plot_df = df.copy()
plot_df["operator_group"] = plot_df["licensee"].map(operator_group)
ops = [op for op in ["Bell", "Rogers", "Bragg", "TELUS", "Other"] if (plot_df["operator_group"] == op).any()]
x = np.arange(len(BANDS))
w = 0.8 / max(len(ops), 1)

counts_by_op = {}
for op in ops:
    sub = plot_df[plot_df.operator_group == op]
    counts_by_op[op] = np.array([
        len(sub[sub.freq_mhz.between(b - 100, b + 100)]) for b in BANDS
    ])

fig, ax = plt.subplots(figsize=(13, 4))

for k, op in enumerate(ops):
    counts = counts_by_op[op]
    offset = (k - (len(ops) - 1) / 2) * w
    bars = ax.bar(x + offset, counts, w, label=op,
                  color=op_color(op), alpha=0.85)
    for bar, count in zip(bars, counts):
        if count > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f"{int(count)}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

max_count = max([counts.max() for counts in counts_by_op.values()] or [1])
ax.set_ylabel("Antenna rows", fontsize=10)
ax.set_ylim(0, max_count * 1.18 + 1)
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in BANDS], fontsize=10)
ax.set_xlabel("Band (MHz)", fontsize=10)
ax.set_title("Antenna deployment by operator - Halifax Peninsula",
             fontweight="bold", fontsize=11)
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig(os.path.join(OUT, "spectrum_deployment.png"), dpi=150, bbox_inches="tight")
plt.show()


### 2.5 Building Height Distribution

Plot the cumulative distribution of building heights, grouped by final height source.


In [ ]:
bld_gj = json.loads(gpd.read_file(BUILDING_HEIGHTS_GPKG).to_crs("EPSG:4326").to_json())
MIN_H = 3.0

#  CDF of building heights by data source 
# Classifies each building by the source of its height value 
# and plots the cumulative distribution for LiDAR, NSTDB/OSM, and default fallback.
# Uses bld_gj (GeoJSON building data)
SOURCE_MAP = {
    "DSM_P95":            "LiDAR (measured)",
    "DSM_DOMINANT":       "LiDAR (measured)",
    "CONSENSUS_DOMINANT": "LiDAR (measured)",
    "NSTDB":              "NSTDB / OSM (inferred)",
    "OSM_NSTDB_CONSENSUS":"NSTDB / OSM (inferred)",
    "OSM":                "NSTDB / OSM (inferred)",
    "DEFAULT_FCODE":      "Default FCODE / fallback",
}

heights_by_src = {"LiDAR (measured)": [], "NSTDB / OSM (inferred)": [], "Default FCODE / fallback": []}
for feat in bld_gj["features"]:
    p   = feat["properties"]
    h   = float(p.get("final_height_m") or 6.5)
    src_key = SOURCE_MAP.get(p.get("final_height_source", "DEFAULT_FCODE"), "Default FCODE / fallback")
    heights_by_src[src_key].append(max(h, MIN_H))

COLORS = {
    "LiDAR (measured)":      "#2ecc71",   # green line
    "NSTDB / OSM (inferred)":"#3498db",   # blue line
    "Default FCODE / fallback":     "#e74c3c",   # red line
}
STYLES = {"LiDAR (measured)": "-", "NSTDB / OSM (inferred)": "--", "Default FCODE / fallback": "-."}

fig, ax = plt.subplots(figsize=(8, 5))
for label, hts in heights_by_src.items():
    if not hts:
        continue
    h_sorted = np.sort(hts)                        
    cdf = np.arange(1, len(h_sorted) + 1) / len(h_sorted)  
    ax.plot(h_sorted, cdf,
            color=COLORS[label], linestyle=STYLES[label],
            label=f"{label} (n={len(hts):,})", linewidth=2)

ax.set_xlabel("Building height (m)")
ax.set_ylabel("CDF")
ax.set_xlim(0, 60)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("CDF of Building Heights — Halifax")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "building_height_cdf.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"LiDAR     : {len(heights_by_src['LiDAR (measured)']):,} buildings")
print(f"NSTDB/OSM : {len(heights_by_src['NSTDB / OSM (inferred)']):,} buildings")
print(f"Default   : {len(heights_by_src['Default FCODE / fallback']):,} buildings")




## 3. Radio Material Model <a id='materials'></a>

Define frequency-dependent ground/material properties used by the radio simulation.


In [ ]:
#  Ground materials ITU-R P.527-3 — 1 MHz to 100 GHz 

# Soil types: B=wet ground, D=medium dry, E=very dry

# Data digitised from ITU-R P.527-3 Figure 1
# Format: (freq_MHz, relative_permittivity_εr, conductivity_σ_S_per_m)

# Curve D — Medium dry ground 
P527_D_DATA = np.array([
    # freq_MHz   εr (rel. permittivity)    σ (conductivity, S/m)
    [1e0,        15.0,    1.0e-3],
    [3e0,        15.0,    1.0e-3],
    [1e1,        15.0,    1.1e-3],
    [3e1,        14.9,    1.5e-3],
    [1e2,        14.5,    3.0e-3],
    [3e2,        13.0,    8.0e-3],
    [1e3,        10.0,    3.5e-2],
    [3e3,         7.0,    1.2e-1],
    [1e4,         5.0,    4.0e-1],
    [3e4,         4.5,    1.5e0 ],
    [1e5,         4.0,    5.0e0 ],
])

# Curve B — Wet ground 
P527_B_DATA = np.array([
    [1e0,        30.0,    1.0e-2],
    [3e0,        30.0,    1.0e-2],
    [1e1,        30.0,    1.1e-2],
    [3e1,        29.5,    1.3e-2],
    [1e2,        28.0,    2.0e-2],
    [3e2,        25.0,    5.0e-2],
    [1e3,        20.0,    1.5e-1],
    [3e3,        15.0,    5.0e-1],
    [1e4,        10.0,    1.5e0 ],
    [3e4,         7.0,    5.0e0 ],
    [1e5,         5.0,    1.5e1 ],
])

# Curve E — Very dry ground 
P527_E_DATA = np.array([
    [1e0,         3.0,    1.0e-4],
    [3e0,         3.0,    1.0e-4],
    [1e1,         3.0,    1.2e-4],
    [3e1,         3.0,    2.0e-4],
    [1e2,         3.0,    5.0e-4],
    [3e2,         3.0,    1.5e-3],
    [1e3,         3.0,    5.0e-3],
    [3e3,         3.0,    1.5e-2],
    [1e4,         3.0,    5.0e-2],
    [3e4,         3.0,    1.5e-1],
    [1e5,         3.0,    5.0e-1],
])

def build_p527_interpolator(data):
    """
    Build a frequency-interpolator for ITU-R P.527-3 ground EM properties.


    data : numpy.ndarray, shape (N, 3)
        ITU-R P.527-3 source table with columns:
            - data[:, 0] : frequency in MHz
            - data[:, 1] : relative permittivity εr  (dimensionless)
            - data[:, 2] : conductivity σ            (S/m)
        Must be sorted in ascending frequency order.
        Typically N=11 points covering 1 MHz to 100 GHz.



    Parameters :
    
    freq_hz : float
        Query frequency in Hz.
        Valid range: 1 MHz  to 100 GHz 
        Outside this range it extrapolates and results
        may be unphysical.
    Returns : 
    er : float
        Relative permittivity εr at freq_hz (dimensionless, > 0).
    sigma : float
        Conductivity σ at freq_hz (S/m, > 0).


    """
    log_f   = np.log10(data[:, 0])   
    log_er  = np.log10(data[:, 1])   
    log_sig = np.log10(data[:, 2])   


    spl_er  = PchipInterpolator(log_f, log_er)
    spl_sig = PchipInterpolator(log_f, log_sig)

    def interp(freq_hz):
        f_mhz  = freq_hz / 1e6        
        log_fm = np.log10(f_mhz)      # convert to log scale for spline evaluation
        er     = float(10 ** spl_er(log_fm))    # back to linear scale 
        sigma  = float(10 ** spl_sig(log_fm))
        return er, sigma

    return interp

# Build the three ITU-R P.527-3 interpolators
p527_medium_dry = build_p527_interpolator(P527_D_DATA)
p527_wet        = build_p527_interpolator(P527_B_DATA)
p527_very_dry   = build_p527_interpolator(P527_E_DATA)

# ── Sanity check 
FREQ_TEST = 700e6
for name, func in [("Medium dry (D)", p527_medium_dry),
                    ("Wet       (B)", p527_wet),
                    ("Very dry  (E)", p527_very_dry)]:
    er, sig = func(FREQ_TEST)
    print(f"  {name} @ {FREQ_TEST/1e9:.1f} GHz : er={er:.3f}  sigma={sig:.5f} S/m")


print("P.527-3 interpolators ready")



In [ ]:
#  Visual check — ITU-R P.527-3 interpolated curves 

f_hz   = np.logspace(6, 11, 500)          # 1 MHz -> 100 GHz
f_mhz  = f_hz / 1e6
f_ghz  = f_hz / 1e9

curves = [
    ("Medium dry (D)", p527_medium_dry, P527_D_DATA, "#2196F3"),
    ("Wet        (B)", p527_wet,        P527_B_DATA, "#4CAF50"),
    ("Very dry   (E)", p527_very_dry,   P527_E_DATA, "#FF5722"),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#1e1e2e")
for ax in (ax1, ax2):
    ax.set_facecolor("#2a2a3e")
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#555")
    ax.grid(True, which="both", color="#444", linewidth=0.5, linestyle="--")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Frequency (GHz)", fontsize=11)

for label, func, raw, color in curves:
    er_interp  = [func(f)[0] for f in f_hz]
    sig_interp = [func(f)[1] for f in f_hz]

    # Interpolated spline curve
    ax1.plot(f_ghz, er_interp,  color=color, lw=2,   label=label)
    ax2.plot(f_ghz, sig_interp, color=color, lw=2,   label=label)

    # Original ITU data points
    pt_f   = raw[:, 0] / 1e3  
    ax1.scatter(pt_f, raw[:, 1], color=color, s=50, zorder=5, edgecolors="white", linewidths=0.6)
    ax2.scatter(pt_f, raw[:, 2], color=color, s=50, zorder=5, edgecolors="white", linewidths=0.6)

# Reference frequency markers (700 MHz, 1.8 GHz, 3.5 GHz, 28 GHz)
for freq_ghz, name in [(0.7, "700M"), (1.8, "1.8G"), (3.5, "3.5G"), (28, "28G")]:
    for ax in (ax1, ax2):
        ax.axvline(freq_ghz, color="yellow", lw=0.8, linestyle=":", alpha=0.7)
        ax.text(freq_ghz*1.05, ax.get_ylim()[0] if ax.get_ylim()[0] > 0 else 1e-5,
                name, color="yellow", fontsize=7, va="bottom")

ax1.set_title("Relative permittivity εr",   fontsize=12)
ax2.set_title("Conductivity σ (S/m)",        fontsize=12)
ax1.set_ylabel("εr (dimensionless)",         fontsize=10, color="white")
ax2.set_ylabel("sigma (S/m)",               fontsize=10, color="white")
ax1.legend(facecolor="#333", labelcolor="white", fontsize=9)
ax2.legend(facecolor="#333", labelcolor="white", fontsize=9)

# Value annotations 
for label, func, _, color in curves:
    f_test = 700e6
    er_test, sig_test = func(f_test)
    print(f"{label:20s} @ {f_test/1e6} MHz : er={er_test:.3f}   sigma={sig_test:.5f} S/m")

fig.suptitle(
    "ITU-R P.527-3 — Log-log interpolation\n"
    "Dots = digitised ITU source data  | Curves = interpolated values",
    color="white", fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()




## 4. Load Sionna Scene <a id='scene'></a>

Load the multi-material Halifax scene from `data/scenes/halifax_peninsula/halifax_peninsula_multi_material.xml` and prepare antenna arrays.


In [ ]:
scene = load_scene(SCENE_XML)


scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="tr38901", polarization="V",
)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern="iso", polarization="V",
)


def set_scene_freq(scene_obj, freq_hz):
    """
    Change scene.frequency safely, bypassing out-of-range ITU material callbacks.
    Materials must be set manually (P.527) after calling this function.
    """
    try:
        scene_obj.frequency = float(freq_hz)
    except (ValueError, Exception):
        # ITU material callback out of valid range (e.g. wet_ground below 1 GHz)
        
        _cbs = {}
        for _n, _m in scene_obj.radio_materials.items():
            _cbs[_n] = getattr(_m, '_frequency_update_callback', None)
            if _cbs[_n] is not None:
                _m._frequency_update_callback = None
        try:
            scene_obj.frequency = float(freq_hz)
        except Exception:
            import drjit as dr
            scene_obj._frequency = dr.Float(float(freq_hz))
        finally:
            for _n, _m in scene_obj.radio_materials.items():
                if _n in _cbs:
                    _m._frequency_update_callback = _cbs[_n]


def set_materials(sc, freq_hz):
    """Apply frequency-dependent EM properties to every object in the scene.

    Iterates over all scene objects and sets their RadioMaterial properties
    according to ITU-R P.527-3 (terrain/ground) or ITU-R P.2040-3
    (building materials: brick, metal, wood, concrete).

    For f >= 1 GHz, Sionna's built-in ITU frequency callbacks update building
    materials automatically; this function only needs to set them explicitly
    for f < 1 GHz (sub-GHz bands where the callback is out of range).
    Terrain is always set explicitly because P.527-3 is not a Sionna callback.

    Parameters :
    
    sc : sionna.rt.Scene
        Loaded Sionna RT scene (must contain a radio_material on each object).
    freq_hz : float
        Current simulation frequency in Hz. Used to compute ITU-R P.527-3
        wet-ground (εr, σ) and to decide whether to apply P.2040-3 manually.
    """
    er_g, sig_g = p527_wet(freq_hz)
    for obj in sc.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat is None:
            continue
        mn = mat.name.lower()
        if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
            # ITU-R P.527-3 wet ground — frequency-dependent
            mat.relative_permittivity, mat.conductivity = er_g, sig_g
        elif freq_hz < 1e9:
            # Sub-GHz: Sionna ITU callback out of range - apply P.2040-3 manually
            if   "brick" in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
            elif "metal" in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
            elif "wood"  in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
            else:               mat.relative_permittivity, mat.conductivity = 5.24, 0.068
        # f >= 1 GHz: Sionna frequency callbacks update ITU materials automatically

set_scene_freq(scene, FREQUENCY)



## 5. Select and Validate Transmitters <a id='tx-selection'></a>

Select a small set of real antenna sectors for the first Sionna simulation.


In [ ]:
#  Antenna selection 
MAX_TX = 10
FREQUENCY_MIN = FREQUENCY / 1e6 - FREQUENCY_WINDOW_MHZ
FREQUENCY_MAX = FREQUENCY / 1e6 + FREQUENCY_WINDOW_MHZ


sim_df = (df[df.freq_mhz.between(FREQUENCY_MIN, FREQUENCY_MAX)]
            .drop_duplicates(subset=["x_local_m", "y_local_m", "azimuth_deg"], keep="first")
            .head(MAX_TX)
            .reset_index(drop=True))

print(f"Selected antennas: {len(sim_df)}")
print(sim_df[["licensee","x_local_m","y_local_m","z_local_m",
              "azimuth_deg","tilt_deg","freq_mhz"]].to_string(index=False))




### 5.2 Transmitter Validation

Validate each selected antenna: simulation bounds, duplicates, co-located sectors, and estimated transmit power.


In [ ]:
#  Validate antennas in sim_df: duplicates, out-of-bounds 
CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

print(f"Validating {len(sim_df)} antennas in sim_df\n")
print(f"  {'tx':<8} {'operator':<12} {'x':>7} {'y':>7} {'z':>6}  {'az':>5}  {'status'}")
print(f"  {'-'*70}")

for i, r in sim_df.iterrows():
    x, y, z = float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])
    op = str(r["licensee"]).split()[0]

    # Out of simulation bounds
    flag = []
    if not (0 <= x <= CM_SIZE_X): flag.append(f"OUT-X ({x:.0f}m)")
    if not (0 <= y <= CM_SIZE_Y): flag.append(f"OUT-Y ({y:.0f}m)")
    if z < 5:                      flag.append(f"SUSPECT-Z ({z:.1f}m)")

    # Exact duplicate (same position and same azimuth+tilt)
    dupes = sim_df[
        (abs(sim_df.x_local_m - r.x_local_m) < 0.1) &
        (abs(sim_df.y_local_m - r.y_local_m) < 0.1) &
        (abs(sim_df.azimuth_deg - r.azimuth_deg) < 0.1) &
        (sim_df.index != i)
    ]
    if len(dupes): flag.append(f"EXACT-DUPLICATE of tx_{dupes.index[0]:03d}")

    # Co-located (same position, different azimuth = different sector)
    coloc = sim_df[
        (abs(sim_df.x_local_m - r.x_local_m) < 0.1) &
        (abs(sim_df.y_local_m - r.y_local_m) < 0.1) &
        (abs(sim_df.azimuth_deg - r.azimuth_deg) >= 0.1) &
        (sim_df.index != i)
    ]
    if len(coloc): flag.append(f"co-located with tx_{coloc.index[0]:03d} (different sector, OK)")

    # Assigned TX transmit power 
    pwr = get_tx_power_dbm(z)

    status = " | ".join(flag) if flag else "OK"
    print(f"  tx_{i:03d}  {op:<12} {x:7.1f} {y:7.1f} {z:6.1f}  {r['azimuth_deg']:5.1f}°"
          f"  {pwr:.0f} dBm  {status}")


print(f"  z >= 20 m → macro        → 43 dBm ")
print(f"  10-20 m   → medium range → 38 dBm")
print(f"  < 10 m    → small cell   → 24 dBm")




## 6. Simple Radio Map <a id='simple-radiomap'></a>

Run one coverage simulation with the selected transmitters. This is the main quick test for the Sionna scene.

**Radio-map height note:** this demo computes the radio map on a fixed horizontal plane at `z = 55 m` in the scene coordinate system. This is an absolute elevation, not `55 m above the terrain`. The local height above ground is approximately `55 m - DEM(x, y)`, so it varies across the peninsula. This is useful for visual comparison, but a user-height coverage study should use receivers placed at `DEM(x, y) + 1.5 m` or another terrain-following height.


In [ ]:
# Coverage map — Transmit Power + P.527-3 materials at current frequency 

# ITU-R P.527-3 materials interpolated at the current frequency 
_freq_hz = float(np.array(scene.frequency).item()) if hasattr(scene.frequency, '__len__') else float(scene.frequency)
er_ground, sig_ground = p527_wet(_freq_hz)
for obj in scene.objects.values():
    mat = getattr(obj, "radio_material", None)
    if mat is None: continue
    mn = mat.name.lower()
    if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
        mat.relative_permittivity, mat.conductivity = er_ground, sig_ground
    elif _freq_hz < 1e9:
        # Sub-GHz : ITU callback not valid - apply P.2040-3 manually
        if "brick"    in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
        elif "metal"  in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
        elif "wood"   in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
        else:                mat.relative_permittivity, mat.conductivity = 5.24, 0.068
    # >= 1 GHz : Sionna updates ITU materials automatically via frequency callback
for name in list(scene.transmitters): scene.remove(name)
for name in list(scene.receivers):    scene.remove(name)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso",     polarization="V")
CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)
for i, r in sim_df.iterrows():
    az_rad   = math.radians(r["azimuth_deg"])
    tilt_rad = math.radians(r["tilt_deg"])
    dist     = 500.0
    pwr_dbm  = get_tx_power_dbm(float(r["z_local_m"]))   
    scene.add(Transmitter(
        name=f"tx_{i:03d}",
        position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
        look_at=[
            float(r["x_local_m"]) + dist * math.sin(az_rad),
            float(r["y_local_m"]) + dist * math.cos(az_rad),
            float(r["z_local_m"]) + dist * math.sin(tilt_rad),
        ],
        power_dbm=pwr_dbm,
    ))
print(f" {len(scene.transmitters)} TX  |  3GPP transmit powers : "
      + ", ".join(f"tx_{i:03d}={get_tx_power_dbm(float(r.z_local_m)):.0f}dBm"
                  for i, r in sim_df.iterrows()))
radiomap = _rm_solver(scene,
     cell_size=(250., 250.),
    # Fixed horizontal demo plane: absolute scene elevation z=55 m, not terrain + 55 m.
    center=[CM_SIZE_X / 2, CM_SIZE_Y / 2, 55.0],
    orientation=[0., 0., 0.],
    size=[CM_SIZE_X, CM_SIZE_Y],
    samples_per_tx=int(2e4),
    max_depth=2,
    los=True, specular_reflection=True,
    edge_diffraction=True,
    diffuse_reflection=True,
)
# path_gain is G (W/W, dimensionless) — convert to dBm: P_TX_dBm + 10*log10(G)
P_TX_dBm_ref = 43.0   
rss_np     = np.array(radiomap.path_gain)
power_grid = rss_np.max(axis=0)
power_dB   = P_TX_dBm_ref + 10 * np.log10(np.clip(power_grid, 1e-20, None))
print(f"Shape      : {power_grid.shape}")
print(f"Cells > 0 : {100*(power_grid>0).mean():.1f}%")
print(f"dBm range  : [{power_dB[power_grid>0].min():.1f}, {power_dB.max():.1f}]")




### 6.2 Coverage Map Visualisation

Render the simple radio map result as a matplotlib coverage panel.


In [ ]:
#  Coverage map visualisation — single panel, dBm scale 

H, W = power_grid.shape

pen = get_peninsula_shape_local()

xx, yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W), np.linspace(0, CM_SIZE_Y, H))
try:
    from shapely.vectorized import contains
    geo_mask = contains(pen, xx.ravel(), yy.ravel()).reshape(H, W)
except (ImportError, AttributeError):
    geo_mask = np.array([pen.contains(Point(xi, yi))
                         for xi, yi in zip(xx.ravel(), yy.ravel())]).reshape(H, W)

DB_MIN, DB_MAX  = -140.0, -30.0
covered         = geo_mask & (power_dB > DB_MIN) & np.isfinite(power_dB)
power_dB_plot   = np.where(covered, np.clip(power_dB, DB_MIN, DB_MAX), np.nan)
coverage_pct    = 100.0 * covered.sum() / geo_mask.sum()

print(f"Peninsula pixels: {geo_mask.sum()}")
print(f"Cells covered : {covered.sum()} ({coverage_pct:.1f}%)")
print(f"Power range (dBm): [{np.nanmin(power_dB_plot):.1f}, {np.nanmax(power_dB_plot):.1f}]")

fig, ax = plt.subplots(1, 1, figsize=(11, 9))
fig.patch.set_facecolor("#0d0d1a")
ax.set_facecolor("#0d0d1a")
for spine in ax.spines.values(): spine.set_edgecolor("#333")

cmap_db = plt.cm.plasma.copy(); cmap_db.set_bad("#0d0d1a")
im = ax.imshow(
    np.ma.masked_invalid(power_dB_plot),
    origin="lower", extent=[0, CM_SIZE_X, 0, CM_SIZE_Y],
    cmap=cmap_db, vmin=DB_MIN, vmax=DB_MAX,
    interpolation="nearest", aspect="equal"
)
ax.set_title(
    f"Coverage Map — Halifax @ {np.array(scene.frequency).item()/1e9:.1f} GHz  ({len(sim_df)} TX)\n"
    f"Coverage : {coverage_pct:.1f}%  |  Max : {np.nanmax(power_dB_plot):.1f} dBm",
    color="white", fontsize=12
)
ax.set_xlabel("East  (m)", color="white")
ax.set_ylabel("North  (m)", color="white")
ax.tick_params(colors="white")
cb = plt.colorbar(im, ax=ax, label="Received power (dBm)", fraction=0.025)
cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")

px, py = pen.exterior.xy

for _, r in sim_df.iterrows():
    ax.plot(r["x_local_m"], r["y_local_m"], "w^", ms=9,
            markeredgecolor="black", markeredgewidth=0.8, zorder=10)

plt.suptitle(
    " Frequency Propagation — Halifax Peninsula\n"
    " max_depth=2  |  diffraction  |  43 dBm TX",
    fontsize=13, fontweight="bold", color="white", y=1.02
)
plt.tight_layout()
out_path = os.path.join(OUT, "coverage_map_halifax.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out_path}")





## 7. Optional Advanced Simulations <a id='advanced'></a>

The following sections are heavier experiments: co-channel SINR, multi-frequency comparisons, and per-band simulations.


### 7.1 Co-channel SINR

Compute the co-channel SINR map. One `RadioMapSolver` run is executed per unique frequency channel.


In [ ]:
#  SINR Map — co-channel interference only 
# Each TX transmits on its real frequency channel 
# Only TX sharing the same channel contribute to G_interf 

BW_Hz = 20e6                               # Bandwidth per channel (assumed)
N0_W  = 10 ** (-174 / 10) * 1e-3 * BW_Hz  # thermal noise = kT·BW at 290 K (−174 dBm/Hz)

CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

CM_PARAMS_22 = dict(
    cell_size=(250., 250.),
    # Fixed horizontal demo plane: absolute scene elevation z=55 m, not terrain + 55 m.
    center=[CM_SIZE_X / 2, CM_SIZE_Y / 2, 55.0],
    orientation=[0., 0., 0.],
    size=[CM_SIZE_X, CM_SIZE_Y],
    samples_per_tx=int(2e4),
    max_depth=2,
    los=True, specular_reflection=True, edge_diffraction=True, diffuse_reflection=True,
)

#  One RadioMapSolver run per real frequency channel 
channel_freqs_22 = sorted(sim_df['freq_mhz'].unique())
raw_by_ch22  = {}   # {freq_mhz: (n_tx_ch, H, W)}
df_by_ch22   = {}
n0r_by_ch22  = {}

print(f"Channel: {[int(f) for f in channel_freqs_22]} MHz  ({len(channel_freqs_22)} simulations)")

for freq_mhz in channel_freqs_22:
    freq_hz = freq_mhz * 1e6
    ch_df   = sim_df[sim_df['freq_mhz'] == freq_mhz].reset_index(drop=True)
    df_by_ch22[freq_mhz] = ch_df

    for name in list(scene.transmitters): scene.remove(name)
    for name in list(scene.receivers):    scene.remove(name)
    set_scene_freq(scene, freq_hz)
    scene.tx_array        = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="V")
    scene.rx_array        = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso",     polarization="V")

    _freq_hz = float(np.array(scene.frequency).item()) if hasattr(scene.frequency, '__len__') else float(scene.frequency)
    er_ground, sig_ground = p527_wet(_freq_hz)
    for obj in scene.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat is None: continue
        mn = mat.name.lower()
        if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
            mat.relative_permittivity, mat.conductivity = er_ground, sig_ground
        elif _freq_hz < 1e9:
            # Sub-GHz: ITU callback not valid → apply P.2040-3 manually
            if "brick"    in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
            elif "metal"  in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
            elif "wood"   in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
            else:                mat.relative_permittivity, mat.conductivity = 5.24, 0.068
        # >= 1 GHz: Sionna updates ITU materials automatically via frequency callback

    p_list = []
    for i, r in ch_df.iterrows():
        az  = math.radians(r["azimuth_deg"])
        tlt = math.radians(r["tilt_deg"])
        pwr = get_tx_power_dbm(float(r["z_local_m"]))
        p_list.append(10 ** (pwr / 10) * 1e-3)
        scene.add(Transmitter(
            name=f"tx_{i:03d}",
            position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
            look_at=[float(r["x_local_m"]) + 500*math.sin(az),
                     float(r["y_local_m"]) + 500*math.cos(az),
                     float(r["z_local_m"]) + 500*math.sin(tlt)],
            power_dbm=pwr,
        ))

    n0r_by_ch22[freq_mhz] = N0_W / float(np.mean(p_list))  # N0 normalised by mean P_TX (same unit as path gain G)
    print(f"  {freq_mhz:.0f} MHz  {len(ch_df)} TX  N0_ratio={n0r_by_ch22[freq_mhz]:.2e} ...", end=" ", flush=True)
    cm = _rm_solver(scene,
     **CM_PARAMS_22)
    raw_by_ch22[freq_mhz] = np.array(cm.path_gain)
    print(f"shape={raw_by_ch22[freq_mhz].shape}")

# Co-channel SINR: best TX per pixel (maximises SINR at each location) 
first = next(iter(raw_by_ch22.values()))
H_s, W_s = first.shape[1], first.shape[2]

sinr_best   = np.full((H_s, W_s), -np.inf)
dominant_tx = np.zeros((H_s, W_s), dtype=int)
g_offset    = 0

for freq_mhz, raw_ch in raw_by_ch22.items():
    N0_ratio = n0r_by_ch22[freq_mhz]
    n_tx_ch  = raw_ch.shape[0]
    for loc in range(n_tx_ch):
        G_sig   = raw_ch[loc]                             
        G_inter = raw_ch.sum(axis=0) - G_sig              # co-channel sum − serving TX = interferers
        s_db    = 10 * np.log10(np.clip(G_sig / (G_inter + N0_ratio), 1e-20, None))  # SINR (dB)
        better  = s_db > sinr_best
        sinr_best[better]   = s_db[better]
        dominant_tx[better] = g_offset + loc
    g_offset += n_tx_ch

sinr_db = sinr_best


pen = get_peninsula_shape_local()
_xx, _yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W_s), np.linspace(0, CM_SIZE_Y, H_s))
try:
    from shapely.vectorized import contains
    geo_mask_sinr = contains(pen, _xx.ravel(), _yy.ravel()).reshape(H_s, W_s)
except (ImportError, AttributeError):
    geo_mask_sinr = np.array([pen.contains(Point(xi, yi))
                               for xi, yi in zip(_xx.ravel(), _yy.ravel())]).reshape(H_s, W_s)

sinr_db_plot     = np.where(geo_mask_sinr, sinr_db,     np.nan)
dominant_tx_plot = np.where(geo_mask_sinr, dominant_tx, -1).astype(float)
dominant_tx_plot[dominant_tx_plot < 0] = np.nan


total = geo_mask_sinr.sum()
print(f"\n--- SINR co-channel ---")
print(f"  SINR > 20 dB  : {100*(geo_mask_sinr & (sinr_db>20)).sum()/total:.1f}%")
print(f"  SINR 10-20 dB : {100*(geo_mask_sinr & (sinr_db>10) & (sinr_db<=20)).sum()/total:.1f}%")
print(f"  SINR  0-10 dB : {100*(geo_mask_sinr & (sinr_db>0)  & (sinr_db<=10)).sum()/total:.1f}%")
print(f"  SINR < 0 dB   : {100*(geo_mask_sinr & (sinr_db<=0)).sum()/total:.1f}%")




In [ ]:
# Channel coverage map — dominant channel per pixel (SINR-based) 
# Dominant channel = the one offering the best SINR at each pixel.
# SINR = G_best_tx / (G_sum_ch - G_best_tx + N0_ratio)  (intra-channel interference only)

from matplotlib.patches import Patch as _Patch

_ch_freqs = sorted(sim_df['freq_mhz'].unique())
_n_ch     = len(_ch_freqs)

if _n_ch < 2:
    print(f"Single channel ({int(_ch_freqs[0])} MHz) — all TX on the same band.")
    print("Widen FREQUENCY_MIN / FREQUENCY_MAX in cell 11 to cover multiple channels,")
    print("then re-run the coverage map cell.")
else:
    _H, _W = rss_np.shape[1], rss_np.shape[2]

    # Noise reference (same assumptions as the SINR cells)
    _BW_Hz    = 20e6
    _P_TX_W   = 10**(P_TX_dBm_ref / 10) * 1e-3
    _N0_W     = 10**(-174/10) * 1e-3 * _BW_Hz
    _N0_ratio = _N0_W / _P_TX_W

    # SINR per channel per pixel
    _sinr_ch = []
    _tx_list = list(sim_df.itertuples())
    for _f in _ch_freqs:
        _idx    = np.array([i for i, r in enumerate(_tx_list) if r.freq_mhz == _f])
        _G_ch   = rss_np[_idx] if len(_idx) > 0 else np.zeros((1, _H, _W))
        _G_best = _G_ch.max(axis=0)
        _G_int  = _G_ch.sum(axis=0) - _G_best   # intra-channel interference
        _sinr_ch.append(_G_best / (_G_int + _N0_ratio))

    _sinr_stack  = np.stack(_sinr_ch, axis=0)   # (N_ch, H, W) linear SINR
    _best_ch_idx = np.argmax(_sinr_stack, axis=0)
    _best_sinr_dB = 10*np.log10(np.clip(_sinr_stack.max(axis=0), 1e-20, None))

    _cmap_tab  = plt.get_cmap("tab10", max(_n_ch, 2))
    _ch_colors = [mcolors.to_hex(_cmap_tab(i)) for i in range(_n_ch)]
    _cmap_ch   = ListedColormap(_ch_colors)

    _ch_map      = np.where(geo_mask, _best_ch_idx.astype(float), np.nan)
    _sinr_plot   = np.where(geo_mask, _best_sinr_dB, np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_facecolor("#0d0d1a")
    for ax in axes:
        ax.set_facecolor("#0d0d1a")
        for spine in ax.spines.values(): spine.set_edgecolor("#333")

    _ext = [0, CM_SIZE_X, 0, CM_SIZE_Y]

    # Left panel: dominant channel (colour = channel)
    im1 = axes[0].imshow(np.ma.masked_invalid(_ch_map),
                         origin="lower", extent=_ext,
                         cmap=_cmap_ch, vmin=-0.5, vmax=_n_ch - 0.5, aspect="equal")
    cb1 = plt.colorbar(im1, ax=axes[0], ticks=np.arange(_n_ch), fraction=0.035)
    cb1.ax.set_yticklabels(
        [f"{int(f)} MHz  ({len(sim_df[sim_df.freq_mhz==f])} TX)" for f in _ch_freqs],
        fontsize=8, color="white")
    cb1.ax.tick_params(colors="white")
    axes[0].set_title("Dominant channel per pixel (best SINR)", color="white", fontsize=11)
    axes[0].set_xlabel("Local Easting (m)", color="white")
    axes[0].set_ylabel("Local Northing (m)", color="white")
    axes[0].tick_params(colors="white", labelsize=8)

    # Right panel: SINR of best channel
    _cmap_sinr = plt.cm.RdYlGn.copy(); _cmap_sinr.set_bad("#0d0d1a")
    im2 = axes[1].imshow(np.ma.masked_invalid(_sinr_plot),
                         origin="lower", extent=_ext,
                         cmap=_cmap_sinr, vmin=-10, vmax=30, aspect="equal")
    cb2 = plt.colorbar(im2, ax=axes[1], label="SINR (dB)", fraction=0.035)
    cb2.ax.yaxis.label.set_color("white"); cb2.ax.tick_params(colors="white")
    axes[1].set_title("SINR of the best channel (dB)", color="white", fontsize=11)
    axes[1].set_xlabel("Local Easting (m)", color="white")
    axes[1].tick_params(colors="white", labelsize=8)

    # TX markers: colour = channel
    for i, _f in enumerate(_ch_freqs):
        _sub = sim_df[sim_df['freq_mhz'] == _f]
        for _, r in _sub.iterrows():
            for ax in axes:
                ax.plot(r["x_local_m"], r["y_local_m"], "^",
                        color=_ch_colors[i], ms=9,
                        markeredgecolor="black", markeredgewidth=0.8, zorder=10)

    _legend_patches = [_Patch(facecolor=_ch_colors[i],
                               label=f"{int(_ch_freqs[i])} MHz  "
                                     f"({len(sim_df[sim_df.freq_mhz==_ch_freqs[i]])} TX)")
                       for i in range(_n_ch)]
    axes[0].legend(handles=_legend_patches, loc="lower left",
                   fontsize=8, framealpha=0.6, labelcolor="white", facecolor="#0d0d1a")

    _suptitle = (f"Channel Coverage Map — Halifax  |  {_n_ch} channels  |  {len(sim_df)} TX  |  "
                 "dominant channel = best SINR per pixel")
    plt.suptitle(_suptitle, fontsize=12, fontweight="bold", color="white", y=1.01)
    plt.tight_layout()
    _out_ch = os.path.join(OUT, "channel_coverage_map.png")
    plt.savefig(_out_ch, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.show()
    print(f"Saved: {_out_ch}")
    for i, _f in enumerate(_ch_freqs):
        _pct = 100 * (geo_mask & (_best_ch_idx == i)).sum() / geo_mask.sum()
        print(f"  {int(_f)} MHz : {_pct:.1f}% of peninsula pixels")



### 7.3 Controlled Frequency Sweep (Same Transmitters)

Run three separate coverage simulations at 700 MHz, 1.8 GHz, and 3.5 GHz using the same transmitter locations. This isolates the effect of frequency from the effect of deployment geometry.


In [ ]:
#  Multi-frequency coverage — dynamic ITU P.527-3 + Transmit power 

FREQUENCIES = {
    "700 MHz (LTE/4G)":    0.7e9,
    "1800 MHz (LTE/4G)":    1.8e9,
    "3500 MHz (5G)": 3.5e9,
}

CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor("#0d0d1a")

for ax, (label, freq) in zip(axes, FREQUENCIES.items()):
    print(f"\n── {label} ──")

    for name in list(scene.transmitters): scene.remove(name)
    for name in list(scene.receivers):    scene.remove(name)

    set_scene_freq(scene, freq)

    # ITU-R P.527-3 materials interpolated at current frequency 
    er_ground,   sig_ground   = p527_wet(freq)
    er_concrete, sig_concrete = 5.31, p527_medium_dry(freq)[1] * 3

    if freq < 1e9:
        print(f"  [sub-GHz] ground : er={er_ground:.2f}  σ={sig_ground:.5f} S/m  "
              f"(vs 3.5 GHz : er={p527_wet(3.5e9)[0]:.2f}  σ={p527_wet(3.5e9)[1]:.5f})")

    mat_names = set()
    for obj in scene.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat: mat_names.add(mat.name)

    _freq_hz = float(np.array(scene.frequency).item()) if hasattr(scene.frequency, '__len__') else float(scene.frequency)
    er_ground, sig_ground = p527_wet(_freq_hz)
    for obj in scene.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat is None: continue
        mn = mat.name.lower()
        if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
            mat.relative_permittivity, mat.conductivity = er_ground, sig_ground
        elif _freq_hz < 1e9:
            # Sub-GHz: ITU callback not valid → apply P.2040-3 manually
            if "brick"    in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
            elif "metal"  in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
            elif "wood"   in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
            else:                mat.relative_permittivity, mat.conductivity = 5.24, 0.068
        # >= 1 GHz: Sionna updates ITU materials automatically via frequency callback

        scene.tx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="V")
        scene.rx_array = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso",     polarization="V")

    for i, r in sim_df.iterrows():
        az_rad   = math.radians(r["azimuth_deg"])
        tilt_rad = math.radians(r["tilt_deg"])
        dist     = 500.0
        scene.add(Transmitter(
            name=f"tx_{i:03d}",
            position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
            look_at=[
                float(r["x_local_m"]) + dist * math.sin(az_rad),
                float(r["y_local_m"]) + dist * math.cos(az_rad),
                float(r["z_local_m"]) + dist * math.sin(tilt_rad),
            ],
            power_dbm=get_tx_power_dbm(float(r["z_local_m"])),   
        ))
    print(f"  {len(scene.transmitters)} TX @ {freq/1e9:.2f} GHz")

    # Larger max_depth at sub-GHz
    _max_depth = 3 if freq < 1e9 else 2

    rm = _rm_solver(scene,
     cell_size=(250., 250.),
        # Fixed horizontal demo plane: absolute scene elevation z=55 m, not terrain + 55 m.
        center=[CM_SIZE_X / 2, CM_SIZE_Y / 2, 55.0],
        orientation=[0., 0., 0.],
        size=[CM_SIZE_X, CM_SIZE_Y],
        samples_per_tx=int(2e4),
        max_depth=_max_depth,
        los=True, specular_reflection=True,
        edge_diffraction=True,
        diffuse_reflection=True,
    )

    rss_np     = np.array(rm.path_gain)
    power_grid = rss_np.max(axis=0)
    power_dB   = 43.0 + 10 * np.log10(np.clip(power_grid, 1e-20, None))
    H, W       = power_grid.shape

    pen = get_peninsula_shape_local()
    _xx, _yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W), np.linspace(0, CM_SIZE_Y, H))
    try:
        from shapely.vectorized import contains
        mask = contains(pen, _xx.ravel(), _yy.ravel()).reshape(H, W)
    except (ImportError, AttributeError):
        mask = np.array([pen.contains(Point(xi, yi))
                         for xi, yi in zip(_xx.ravel(), _yy.ravel())]).reshape(H, W)

    DB_MIN, DB_MAX = -140.0, -30.0
    covered     = mask & (power_dB > DB_MIN) & np.isfinite(power_dB)
    plot        = np.where(covered, np.clip(power_dB, DB_MIN, DB_MAX), np.nan)
    coverage_pct = 100 * covered.sum() / mask.sum()
    mean_rsrp    = float(power_dB[covered].mean()) if covered.sum() > 0 else 0
    print(f"  Coverage: {coverage_pct:.1f}% ")

    ax.set_facecolor("#0d0d1a")
    for spine in ax.spines.values(): spine.set_edgecolor("#333")
    cmap = plt.cm.inferno.copy(); cmap.set_bad("#0d0d1a")
    im = ax.imshow(np.ma.masked_invalid(plot), origin="lower",
                   extent=[0, CM_SIZE_X, 0, CM_SIZE_Y],
                   cmap=cmap, vmin=DB_MIN, vmax=DB_MAX, aspect="equal")
    cb = plt.colorbar(im, ax=ax, label="RSRP (dBm)", fraction=0.035)
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    ax.set_title(
        f"{label}\nmax_depth={_max_depth} \n"
        f"Cov. {coverage_pct:.0f}%  |  mean RSRP {mean_rsrp:.0f} dBm",
        color="white", fontsize=9)
    ax.set_xlabel("Local Easting (m)", color="white")
    ax.set_ylabel("Local Northing (m)", color="white")
    ax.tick_params(colors="white", labelsize=8)
    for _, r in sim_df.iterrows():
        ax.plot(r["x_local_m"], r["y_local_m"], "w^", ms=7,
                markeredgecolor="black", markeredgewidth=0.8, zorder=10)

set_scene_freq(scene, FREQUENCY)
print(f"\n Restored -> {FREQUENCY/1e6:.0f} MHz")

plt.suptitle(
    "Controlled Frequency Sweep - Same Transmitters\n",
    fontsize=13, fontweight="bold", color="white", y=1.01)
plt.tight_layout()
out_mf = os.path.join(OUT, "coverage_multifreq.png")
plt.savefig(out_mf, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out_mf}")




### 7.4 Real Per-band Antenna Simulations

Run per-band coverage simulations using the real antennas that operate in each band.


In [ ]:
#  Multi-frequency coverage — real per-band antennas from  dataset 
# Free memory from previous computations 
gc.collect()
print("Memory freed")
MAX_TX = 15   # max antennas per band

#  Select antennas from df (peninsula_cellular_antennas_2g_to_5g.csv) 


def get_antennas_for_band(freq_min, freq_max, max_tx=MAX_TX):
    """Return up to max_tx antennas in [freq_min, freq_max] MHz from df.

    If more than max_tx antennas exist in the band, a spatial grid
    sub-sampling is applied: the bounding box is divided into n_side x n_side
    cells and the antenna closest to each cell centre is selected, ensuring
    geographic spread across the peninsula rather than clustering.

    Parameters
    ----------
    freq_min, freq_max : float
        Frequency band limits in MHz.
    max_tx : int
        Maximum number of transmitters to return.

    Returns
    -------
    pandas.DataFrame
        Subset of df with at most max_tx rows, reset index.
    """
    df_band = df[df["freq_mhz"].between(freq_min, freq_max)].copy()
    if len(df_band) <= max_tx:
        return df_band.reset_index(drop=True)

    # Spatial grid: divide the bbox into cells, pick antenna closest to each centre
    coords  = df_band[["x_local_m", "y_local_m"]].values
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    n_side  = int(np.ceil(np.sqrt(max_tx)))
    xs      = np.linspace(x_min, x_max, n_side + 1)
    ys      = np.linspace(y_min, y_max, n_side + 1)
    selected = set()
    for gi in range(n_side):
        for gj in range(n_side):
            cx = (xs[gi] + xs[gi+1]) / 2
            cy = (ys[gj] + ys[gj+1]) / 2
            dists = np.sqrt((coords[:, 0] - cx)**2 + (coords[:, 1] - cy)**2)
            selected.add(int(np.argmin(dists)))
            if len(selected) >= max_tx:
                break
        if len(selected) >= max_tx:
            break
    return df_band.iloc[sorted(selected)].reset_index(drop=True)

# Frequency bands 
# Each entry: display label - (centre frequency Hz, band min MHz, band max MHz)
FREQUENCIES = {
    "700 MHz (LTE)" : (0.700e9, 600,  800),
    "2100 MHz (LTE)" : (2.100e9, 2000, 2200),
    "3500 MHz (5G)"  : (3.500e9, 3300, 3700),
}

# Preview antenna counts before running the RadioMapSolver loop
for label, (freq_center, f_min, f_max) in FREQUENCIES.items():
    df_check = get_antennas_for_band(f_min, f_max, max_tx=9999)
    print(f"{label}: {len(df_check)} antennas → {min(len(df_check), MAX_TX)} selected")

#  Fallback materials
# These are static (frequency-independent) approximations used as a safety net.
try:
    mat_ground   = RadioMaterial("mf_ground",   relative_permittivity=15.0, conductivity=0.005)
    mat_concrete = RadioMaterial("mf_concrete", relative_permittivity=5.31, conductivity=0.326)
except Exception:
    mat_ground   = scene.get("mf_ground")
    mat_concrete = scene.get("mf_concrete")

# This pre-loop pass sets physically accurate EM properties on every scene object.
# For f >= 1 GHz, Sionna updates ITU materials automatically via frequency
_freq_hz = float(np.array(scene.frequency).item()) if hasattr(scene.frequency, '__len__') else float(scene.frequency)
er_ground, sig_ground = p527_wet(_freq_hz)
for obj in scene.objects.values():
    mat = getattr(obj, "radio_material", None)
    if mat is None: continue
    mn = mat.name.lower()
    if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
        mat.relative_permittivity, mat.conductivity = er_ground, sig_ground
    elif _freq_hz < 1e9:
        # Sub-GHz: ITU callback not valid → apply P.2040-3 Table 3 manually
        if "brick"    in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
        elif "metal"  in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
        elif "wood"   in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
        else:                mat.relative_permittivity, mat.conductivity = 5.24, 0.068

#  Scene bounding box 
CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

_pen = get_peninsula_shape_local()

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor("#0d0d1a")

#  one RadioMapSolver run per frequency band 
# loop so each band gets its own simulation and subplot.
for ax, (label, (freq_hz, f_min, f_max)) in zip(axes, FREQUENCIES.items()):
    print(f"\n── {label} ──")

    # Select geographically-spread antennas for this band
    df_band = get_antennas_for_band(f_min, f_max, max_tx=MAX_TX)
    if len(df_band) == 0:
        print(f"  No antennas found for {label}")
        ax.set_facecolor("#0d0d1a")
        ax.set_title(f"{label}\nNo data", color="white", fontsize=9)
        continue
    print(f"  {len(df_band)} antennas selected")

    # Configure scene for this frequency band 
    for name in list(scene.transmitters): scene.remove(name)
    for name in list(scene.receivers):    scene.remove(name)
    set_scene_freq(scene, freq_hz)

    # Apply per-material-type ITU-R P.2040-3/P.527-3 EM properties
    set_materials(scene, freq_hz)

    
    scene.tx_array = PlanarArray(
        num_rows=1, num_cols=1,
        vertical_spacing=0.5, horizontal_spacing=0.5,
        pattern="tr38901", polarization="V",
    )
    scene.rx_array = PlanarArray(
        num_rows=1, num_cols=1,
        vertical_spacing=0.5, horizontal_spacing=0.5,
        pattern="iso", polarization="V",
    )

    #  Place transmitters using real azimuth and downtilt 
    for idx, r in df_band.iterrows():
        az_rad   = math.radians(r["azimuth_deg"])
        tilt_rad = math.radians(r["tilt_deg"])    
        dist     = 500.0
        scene.add(Transmitter(
            name=f"tx_{idx:03d}",
            position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
            look_at=[
                float(r["x_local_m"]) + dist * math.sin(az_rad),
                float(r["y_local_m"]) + dist * math.cos(az_rad),
                float(r["z_local_m"]) + dist * math.sin(tilt_rad),
            ],
            power_dbm=43.0,
        ))

    #  Run RadioMapSolver 

    rm = _rm_solver(
        scene,
        cell_size=(250., 250.),
        # Fixed horizontal demo plane: absolute scene elevation z=55 m, not terrain + 55 m.
        center=[CM_SIZE_X / 2, CM_SIZE_Y / 2, 55.0],
        orientation=[0., 0., 0.],
        size=[CM_SIZE_X, CM_SIZE_Y],
        samples_per_tx=int(2e4),
        max_depth=2,
        los=True, specular_reflection=True,
        edge_diffraction=True,
        diffuse_reflection=True,
    )

    #  path_gain (linear) → RSRP in dBm 
    # path_gain is dimensionless
    # max over TX axis gives the best-server (strongest signal) per pixel.
    rss_np     = np.array(rm.path_gain)              # shape: (N_TX, H, W)
    power_grid = rss_np.max(axis=0)                  # best-server map (linear)
    power_dB   = 10 * np.log10(np.clip(power_grid, 1e-20, None)) + 30  # dBm
    H, W       = power_grid.shape

    _xx, _yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W), np.linspace(0, CM_SIZE_Y, H))
    try:
        from shapely.vectorized import contains
        mask = contains(_pen, _xx.ravel(), _yy.ravel()).reshape(H, W)
    except (ImportError, AttributeError):
        mask = np.array([_pen.contains(Point(xi, yi))
                         for xi, yi in zip(_xx.ravel(), _yy.ravel())]).reshape(H, W)

    #  Coverage metrics 
    DB_MIN, DB_MAX = -140.0, -30.0
    covered   = mask & (power_dB > DB_MIN) & np.isfinite(power_dB)
    plot      = np.where(covered, np.clip(power_dB, DB_MIN, DB_MAX), np.nan)
    cov_pct   = 100 * covered.sum() / mask.sum()
    mean_rsrp = float(power_dB[covered].mean()) if covered.sum() > 0 else 0.0
    print(f"  Coverage: {cov_pct:.1f}%  |  mean RSRP: {mean_rsrp:.1f} dBm")

    #  Render RSRP heatmap 
    ax.set_facecolor("#0d0d1a")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    cmap = plt.cm.inferno.copy()
    cmap.set_bad("#0d0d1a")
    im = ax.imshow(np.ma.masked_invalid(plot), origin="lower",
                   extent=[0, CM_SIZE_X, 0, CM_SIZE_Y],
                   cmap=cmap, vmin=DB_MIN, vmax=DB_MAX, aspect="equal")
    cb = plt.colorbar(im, ax=ax, label="RSRP (dBm)", fraction=0.035)
    cb.ax.yaxis.label.set_color("white")
    cb.ax.tick_params(colors="white")
    ax.set_title(
        f"{label}\n{len(df_band)} real antennas  |  Cov. {cov_pct:.0f}%  |  mean {mean_rsrp:.0f} dBm",
        color="white", fontsize=9)
    ax.set_xlabel("Local Easting (m)", color="white")
    ax.set_ylabel("Local Northing (m)", color="white")
    ax.tick_params(colors="white", labelsize=8)

    # Mark TX antenna positions with white triangles
    for _, r in df_band.iterrows():
        ax.plot(r["x_local_m"], r["y_local_m"], "w^", ms=6,
                markeredgecolor="black", markeredgewidth=0.6, zorder=10)

#  Restore scene to 5G frequency after the multi-band loop 
set_scene_freq(scene, FREQUENCY)

plt.suptitle(
    "Real Per-band Coverage Map - Halifax Peninsula\n"
    "Band-specific antenna rows  |  fixed P_tx=43 dBm ",
    fontsize=13, fontweight="bold", color="white", y=1.01)
plt.tight_layout()
out_mf = os.path.join(OUT, "coverage_multifreq_real.png")
plt.savefig(out_mf, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: {out_mf}")




## 8. Radio Metrics <a id='metrics'></a>

Compute summary radio metrics from the simulated coverage maps.


In [ ]:

gc.collect()


#  TX selection 
MAX_TX_SINR = 15
sim_df = (df[df.freq_mhz.between(FREQUENCY_MIN, FREQUENCY_MAX)]
               .head(MAX_TX_SINR)
               .reset_index(drop=True))
channel_freqs = sorted(sim_df['freq_mhz'].unique())
print(f"Selected TX : {len(sim_df)}")
print(f"Channels : {[int(f) for f in channel_freqs]} MHz  ({len(channel_freqs)} simulations)")

CM_SIZE_X = float(X_MAX)
CM_SIZE_Y = float(Y_MAX)

N_RUNS      = 5              
SAMPLES_RUN = int(2e4)       # CPU-preview rays per run; increase for publication-quality maps
NF_dB       = 9.0            # receiver noise figure (dB) — raises the effective noise floor
BW_Hz    = 20e6           # bandwidth in Hz 
N0_W     = 10**(-174/10) * 1e-3 * BW_Hz * 10**(NF_dB/10)  # noise power in W

cm_params = dict(
    cell_size=(250., 250.),
    # Fixed horizontal demo plane: absolute scene elevation z=55 m, not terrain + 55 m.
    center=[CM_SIZE_X/2, CM_SIZE_Y/2, 55.0],
    orientation=[0., 0., 0.],
    size=[CM_SIZE_X, CM_SIZE_Y],
    samples_per_tx=SAMPLES_RUN,
    max_depth=2,
    los=True, specular_reflection=True, edge_diffraction=True, diffuse_reflection=True,
)

#  Simulation per channel 
raw_by_channel = {}   # {freq_mhz: (n_tx_ch, H, W)}
df_by_channel  = {}
n0r_by_ch      = {}

for freq_mhz in channel_freqs:
    freq_hz = freq_mhz * 1e6
    ch_df   = sim_df[sim_df['freq_mhz'] == freq_mhz].reset_index(drop=True)
    df_by_channel[freq_mhz] = ch_df
    print(f"\n-- Channel {freq_mhz:.0f} MHz ({len(ch_df)} TX) --")

    for name in list(scene.transmitters): scene.remove(name)
    for name in list(scene.receivers):    scene.remove(name)
    set_scene_freq(scene, freq_hz)
    scene.tx_array        = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="tr38901", polarization="V")
    scene.rx_array        = PlanarArray(num_rows=1, num_cols=1, vertical_spacing=0.5, horizontal_spacing=0.5, pattern="iso",     polarization="V")

    _freq_hz = float(np.array(scene.frequency).item()) if hasattr(scene.frequency, '__len__') else float(scene.frequency)
    er_ground, sig_ground = p527_wet(_freq_hz)
    for obj in scene.objects.values():
        mat = getattr(obj, "radio_material", None)
        if mat is None: continue
        mn = mat.name.lower()
        if "terrain" in mn or "ground" in mn or "wet_ground" in mn:
            mat.relative_permittivity, mat.conductivity = er_ground, sig_ground
        elif _freq_hz < 1e9:
            # Sub-GHz: ITU callback not valid → apply P.2040-3 manually
            if "brick"    in mn: mat.relative_permittivity, mat.conductivity = 3.91, 0.020
            elif "metal"  in mn: mat.relative_permittivity, mat.conductivity = 1.00, 1e7
            elif "wood"   in mn: mat.relative_permittivity, mat.conductivity = 1.99, 0.004
            else:                mat.relative_permittivity, mat.conductivity = 5.24, 0.068
        # >= 1 GHz: Sionna updates ITU materials automatically via frequency callback

    p_list = []
    for i, r in ch_df.iterrows():
        az  = math.radians(r["azimuth_deg"])
        tlt = math.radians(r["tilt_deg"])
        pwr = get_tx_power_dbm(float(r["z_local_m"]))
        p_list.append(10**(pwr/10)*1e-3)
        scene.add(Transmitter(
            name=f"tx_{i:03d}",
            position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
            look_at=[float(r["x_local_m"]) + 500*math.sin(az),
                     float(r["y_local_m"]) + 500*math.cos(az),
                     float(r["z_local_m"]) + 500*math.sin(tlt)],
            power_dbm=pwr,
        ))
    n0r_by_ch[freq_mhz] = N0_W / float(np.mean(p_list))
    print(f"  N0_ratio={n0r_by_ch[freq_mhz]:.2e}  |  {N_RUNS} runs x {SAMPLES_RUN:,} samples")

    acc = None
    for run in range(N_RUNS):
        print(f"  Run {run+1}/{N_RUNS}...", end=" ", flush=True)
        cm_run  = _rm_solver(scene,
     **cm_params)
        rss_run = np.array(cm_run.path_gain)
        acc = rss_run.copy() if acc is None else acc + rss_run
        del cm_run, rss_run
        print("OK")
    raw_by_channel[freq_mhz] = acc / N_RUNS
    gc.collect()

first_raw = next(iter(raw_by_channel.values()))
H_s, W_s = first_raw.shape[1], first_raw.shape[2]
pen = get_peninsula_shape_local()
_xx, _yy = np.meshgrid(np.linspace(0, CM_SIZE_X, W_s), np.linspace(0, CM_SIZE_Y, H_s))
try:
    from shapely.vectorized import contains
    mask = contains(pen, _xx.ravel(), _yy.ravel()).reshape(H_s, W_s)
except (ImportError, AttributeError):
    mask = np.array([pen.contains(Point(xi, yi))
                        for xi, yi in zip(_xx.ravel(), _yy.ravel())]).reshape(H_s, W_s)

#   Co-channel radio metrics 
# RSRP  : best TX across all channels combined
# RSSI  : sum of co-channel TX power + noise (serving channel)
# SINR  : G_signal / (G_interf_cochannel + N0_ratio)

raw = np.concatenate(
    [raw_by_channel[f] for f in channel_freqs], axis=0)  # (N_tot, H, W)

P_TX_dBm_ref = 43.0
P_TX_W_ref   = 10**(P_TX_dBm_ref/10)*1e-3
N0_ratio_ref = N0_W / P_TX_W_ref
N0_dBm       = 10 * np.log10(N0_W) + 30

RSRP_dBm  = P_TX_dBm_ref + 10 * np.log10(np.clip(raw.max(axis=0), 1e-20, None))

sinr_best = np.full((H_s, W_s), -np.inf)
rssi_ch   = np.zeros((H_s, W_s))

for freq_mhz, raw_ch in raw_by_channel.items():
    N0_ratio = n0r_by_ch[freq_mhz]
    G_sum_ch = raw_ch.sum(axis=0)
    for loc in range(raw_ch.shape[0]):
        G_sig   = raw_ch[loc]
        G_inter = G_sum_ch - G_sig
        s_db    = 10 * np.log10(np.clip(G_sig / (G_inter + N0_ratio), 1e-20, None))
        better  = s_db > sinr_best
        sinr_best[better] = s_db[better]
        rssi_ch[better]   = G_sum_ch[better]

SINR_dB  = sinr_best
RSSI_dBm = P_TX_dBm_ref + 10 * np.log10(np.clip(rssi_ch + N0_ratio_ref, 1e-20, None))

n_tx        = raw.shape[0]
RSRP_MIN    = -110.0
signal_mask = mask & (RSRP_dBm > RSRP_MIN) & np.isfinite(RSRP_dBm)
cov_pct     = 100 * signal_mask.sum() / mask.sum()

print(f"\n{'='*62}")
print(f"  {n_tx} TX | NF={NF_dB} dB | co-channel")
print(f"  Covered area (>{RSRP_MIN} dBm) : {cov_pct:.1f}%")
print(f"  N0 : {N0_dBm:.1f} dBm")
print(f"{'='*62}")
print(f"  {'Metric':<12} {'Min':>7} {'Median':>9} {'Average':>8} {'Max':>7}")
print(f"  {'-'*48}")
for mname, arr in [("RSRP_dBm", RSRP_dBm), ("RSSI_dBm", RSSI_dBm), ("SINR_dB", SINR_dB)]:
    v = arr[signal_mask]
    if len(v):
        print(f"  {mname:<12} {v.min():>7.1f} {np.median(v):>9.1f} {v.mean():>8.1f} {v.max():>7.1f}")
print(f"{'='*62}\n")

#  Maps 
extent = [0, CM_SIZE_X, 0, CM_SIZE_Y]
configs = {
    "RSRP (dBm)":         (RSRP_dBm, "plasma",  -140, -50),
    "SINR co-channel (dB)": (SINR_dB,  "RdYlGn",   -5,  30),
}
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#0d0d1a")
for ax, (label, (arr, cmap_name, vmin, vmax)) in zip(axes, configs.items()):
    ax.set_facecolor("#0d0d1a")
    for spine in ax.spines.values(): spine.set_edgecolor("#333")
    cmap = plt.get_cmap(cmap_name).copy(); cmap.set_bad("#0d0d1a")
    data = np.where(signal_mask, arr, np.nan)
    im = ax.imshow(np.ma.masked_invalid(data), origin="lower", extent=extent,
                   cmap=cmap, vmin=vmin, vmax=vmax, aspect="equal")
    cb = plt.colorbar(im, ax=ax, label=label, fraction=0.035)
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    ax.set_title(f"{label}\n{FREQUENCY/1e9:.3g} GHz | {N_RUNS}x{SAMPLES_RUN/1e6:.0f}M samples | co-channel",
                 fontsize=10, color="white", pad=6)
    ax.set_xlabel("East (m)", color="white")
    ax.set_ylabel("North (m)", color="white")
    ax.tick_params(colors="white", labelsize=8)
    px, py = pen.exterior.xy
    ax.plot(px, py, color="white", linewidth=0.7, alpha=0.5)
    for _, r in sim_df.iterrows():
        ax.plot(r["x_local_m"], r["y_local_m"], "w^", ms=5,
                markeredgecolor="black", markeredgewidth=0.5, zorder=10)

plt.suptitle(f"RSRP & SINR co-channel -- Halifax | {n_tx} TX  | NF={NF_dB} dB",
             fontsize=12, fontweight="bold", color="white")
plt.tight_layout()
out = os.path.join(OUT, "radio_metrics_maps.png")
plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"  {out}")




In [ ]:
#  Extract power_maps + channel_map from coverage map results 
#  (N_tot, H, W) — concatenation of all channels
# channel_map= {tx_name: freq_mhz} — enables co-channel SINR in compute_radio_metrics

n_tx_total    = raw.shape[0]
tx_names_used = [f'tx_{i:03d}' for i in range(n_tx_total)]
power_maps    = {tx_names_used[i]: raw[i] for i in range(n_tx_total)}
geo_mask      = mask

sim_df_ordered = pd.concat(
    [sim_df[sim_df['freq_mhz'] == f].reset_index(drop=True)
     for f in sorted(sim_df['freq_mhz'].unique())],
    ignore_index=True
)
channel_map = {
    f'tx_{i:03d}': float(sim_df_ordered.iloc[i]['freq_mhz'])
    for i in range(len(sim_df_ordered))
}

display_metrics = {'RSSI_dBm': RSSI_dBm, 'SINR_dB': SINR_dB}

n_ch = len(set(channel_map.values()))
print(f'power_maps     : {len(power_maps)} TX  |  shape : {raw[0].shape}')
print(f'channel_map : {n_ch} distinct channel(s)')
print('  ' + '  |  '.join(
    [f'{int(f)} MHz: {sum(1 for v in channel_map.values() if v==f)} TX'
     for f in sorted(set(channel_map.values()))]))
print(f'geo_mask       : {geo_mask.sum():,} pixels inside the peninsula')
print(f'Covered area : {cov_pct:.1f}%')




### 8.2 RSRP, RSSI, and SINR Metrics

Define and run `compute_radio_metrics(power_maps, channel_map)`.


In [ ]:
# power_maps  : {tx_name: (H,W) path gain G (W/W)}
# channel_map : {tx_name: channel_id}  -- if None: all TX treated as co-channel
#
# RSRP = P_TX_dBm + 10*log10(G_best)                  best TX across all channels
# RSSI = P_TX_dBm + 10*log10(G_sum_cochannel + N0_ratio) serving channel
# SINR = G_signal / (G_interf_cochannel + N0_ratio)       co-channel only

P_TX_dBm = 43.0
P_TX_W   = 10**(P_TX_dBm / 10) * 1e-3
BW_Hz    = 20e6
N0_W     = 10**(-174/10) * 1e-3 * BW_Hz
N0_ratio = N0_W / P_TX_W
N0_dBm   = P_TX_dBm + 10 * np.log10(N0_ratio)

def compute_radio_metrics(power_maps, channel_map=None):
    '''power_maps  : dict {tx_name: (H,W) path gain G}
    channel_map : dict {tx_name: channel_id}  
    '''
    tx_names = list(power_maps.keys())
    G = np.stack(list(power_maps.values()), axis=0)   # (N_TX, H, W)

    G_max    = G.max(axis=0)
    RSRP_dBm = P_TX_dBm + 10 * np.log10(np.clip(G_max, 1e-20, None))

    if channel_map is None:
        G_sum    = G.sum(axis=0)
        G_interf = G_sum - G_max
        RSSI_dBm = P_TX_dBm + 10 * np.log10(np.clip(G_sum    + N0_ratio, 1e-20, None))
        SINR_dB  = 10 * np.log10(np.clip(G_max / (G_interf + N0_ratio), 1e-20, None))
    else:
        channels  = np.array([channel_map[n] for n in tx_names])
        sinr_best = np.full(G_max.shape, -np.inf)
        rssi_ch   = np.zeros(G_max.shape)
        for ch in np.unique(channels):
            idx_ch   = np.where(channels == ch)[0]
            G_ch     = G[idx_ch]
            G_sum_ch = G_ch.sum(axis=0)
            for k, gi in enumerate(idx_ch):
                G_sig   = G[gi]
                G_inter = G_sum_ch - G_sig
                s_db    = 10 * np.log10(np.clip(G_sig / (G_inter + N0_ratio), 1e-20, None))
                better  = s_db > sinr_best
                sinr_best[better] = s_db[better]
                rssi_ch[better]   = G_sum_ch[better]
        RSSI_dBm = P_TX_dBm + 10 * np.log10(np.clip(rssi_ch + N0_ratio, 1e-20, None))
        SINR_dB  = sinr_best

    return {'RSRP_dBm': RSRP_dBm, 'RSSI_dBm': RSSI_dBm, 'SINR_dB': SINR_dB}

#  Recalculation 
if 'channel_map' in dir() and channel_map:
    n_ch = len(set(channel_map.values()))
    print(f'compute_radio_metrics with channel_map ({n_ch} channels) -- co-channel SINR')
    metrics = compute_radio_metrics(power_maps, channel_map=channel_map)
else:
    print('compute_radio_metrics without channel_map (all co-channel)')
    metrics = compute_radio_metrics(power_maps)

signal_mask = geo_mask & (metrics['RSRP_dBm'] > (N0_dBm + 10))
display_metrics = {'RSSI_dBm': metrics['RSSI_dBm'], 'SINR_dB': metrics['SINR_dB']}
cov_pct = 100 * signal_mask.sum() / geo_mask.sum()
print(f'Covered area (peninsula) : {cov_pct:.1f}%')
for mname, arr in display_metrics.items():
    v = arr[signal_mask]
    print(f'  {mname:<10}  min={v.min():.1f}  med={np.median(v):.1f}  '
          f'mean={v.mean():.1f}  max={v.max():.1f}')




In [ ]:
def compare_metrics_by_freq(power_maps_dict, geo_mask, channel_maps=None):
    """Compare radio metrics (RSRP/RSSI/SINR) across frequency bands.
    Parameters: power_maps_dict={label:power_maps}, geo_mask, channel_maps={label:channel_map} (optional).
    """
    channel_maps = channel_maps or {}
    print(f"\n{'Frequency':<26} {'RSSI med':>10} {'RSSI>-95':>10} {'SINR med':>10} {'SINR>0dB':>10} {'Cov.':>7}")
    print('-' * 78)
    results = {}
    for freq_label, pm in power_maps_dict.items():
        ch_map = channel_maps.get(freq_label)
        m      = compute_radio_metrics(pm, channel_map=ch_map)
        H, W   = m['RSRP_dBm'].shape
        if geo_mask.shape != (H, W):
            from scipy.ndimage import zoom
            gm = (zoom(geo_mask.astype(np.float32),
                       (H / geo_mask.shape[0], W / geo_mask.shape[1])) > 0.5)
        else:
            gm = geo_mask
        mask   = gm & (m['RSRP_dBm'] > (N0_dBm + 10))
        rssi   = m['RSSI_dBm'][mask]
        sinr   = m['SINR_dB'][mask]
        cov    = 100 * mask.sum() / gm.sum()
        tag    = ' (co-ch)' if ch_map else ''
        print(f'  {freq_label:<24} {np.median(rssi):>9.1f} dBm'
              f'  {100*(rssi > -95).mean():>8.1f}%'
              f'  {np.median(sinr):>8.1f} dB{tag}'
              f'  {100*(sinr > 0).mean():>8.1f}%'
              f'  {cov:>5.1f}%')
        results[freq_label] = dict(
            rssi_med=np.median(rssi), sinr_med=np.median(sinr),
            rssi_95=100*(rssi>-95).mean(), sinr_0=100*(sinr>0).mean(), cov=cov)
    return results

if 'power_maps_by_freq' not in dir():
    power_maps_by_freq = {'3500 MHz (5G)': power_maps}

ch_maps_arg = {}
if 'channel_map' in dir() and channel_map:
    for lbl in power_maps_by_freq:
        ch_maps_arg[lbl] = channel_map

freq_results = compare_metrics_by_freq(power_maps_by_freq, geo_mask, channel_maps=ch_maps_arg)




## 9. 3D Preview and Ray Paths <a id='ray-paths'></a>

Optional 3D inspection of the scene, transmitters, receivers, and ray paths.


In [ ]:
#  Sionna 3D preview — full scene with TX/RX 
# RF materials (permittivity, conductivity) defined in the previous cells.
scene.preview(show_devices=True)



### 9.2 PathSolver Ray Tracing

Run `PathSolver` to compute ray paths between selected transmitters and receivers.


In [ ]:

# Runs PathSolver to compute
# the actual ray paths between TX antennas and a grid of RX positions.
# Then renders the paths in 3D using scene.preview().
#
# Setup:
#   - N_TX_RT=5 transmitters taken from sim_df (real Bell Mobility positions)
#   - N_RX_PER_TX=4 receivers per TX, placed RX_DIST=300 m away
#   - RX height: RX_HT=55 m 
#   - TX array: tr38901 pattern, RX array: isotropic
#
# PathSolver outputs :
#   paths.vertices   — 3D coordinates of each path vertex (reflection/diff points)
#   paths.types      — LOS, reflected, diffracted, scattered
#   paths.a          — channel coefficients
#
#  Ray Tracing — real TX/RX positions + 3D visualisation 


import math, numpy as np
from sionna.rt import Transmitter, Receiver, PlanarArray

RX_HT       = 55.0       # Absolute scene elevation for the demo receivers (m) 
N_TX_RT     = min(5, len(sim_df))
N_RX_PER_TX = 4
RX_DIST     = 300.0

for name in list(scene.transmitters): scene.remove(name)
for name in list(scene.receivers):    scene.remove(name)

scene.tx_array = PlanarArray(
    num_rows=1, num_cols=1, vertical_spacing=0.5,
    horizontal_spacing=0.5, pattern="tr38901", polarization="V"
)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1, vertical_spacing=0.5,
    horizontal_spacing=0.5, pattern="iso", polarization="V"
)

rx_global_idx = 0
for i, r in sim_df.head(N_TX_RT).iterrows():
    az_rad   = math.radians(r["azimuth_deg"])
    tilt_rad = math.radians(r["tilt_deg"])
    dist     = 500.0
    scene.add(Transmitter(
        name=f"tx_{i:03d}",
        position=[float(r["x_local_m"]), float(r["y_local_m"]), float(r["z_local_m"])],
        look_at=[
            float(r["x_local_m"]) + dist * math.sin(az_rad),
            float(r["y_local_m"]) + dist * math.cos(az_rad),
            float(r["z_local_m"]) + dist * math.sin(tilt_rad),
        ],
        power_dbm=43.0,
    ))
    for k in range(N_RX_PER_TX):
        angle = az_rad + math.radians(-90 + k * 180 / (N_RX_PER_TX - 1))
        rx_x  = float(r["x_local_m"]) + RX_DIST * math.sin(angle)
        rx_y  = float(r["y_local_m"]) + RX_DIST * math.cos(angle)
        scene.add(Receiver(
            name=f"rx_{rx_global_idx:03d}", position=[rx_x, rx_y, RX_HT]
        ))
        rx_global_idx += 1

paths = _path_solver(scene,
    max_depth=2, samples_per_src=int(2e6),
    los=True, specular_reflection=True, edge_diffraction=True, diffuse_reflection=True,
)

# Count valid paths 
import drjit as dr
total_paths = 0
try:
    # paths.objects contains the computed ray paths
    tau_arr = np.array(paths.tau)           # (1, n_rx, n_tx, max_paths)
    a_arr   = np.abs(np.array(paths.a)) 
    
    valid_mask = (tau_arr > 1e-10) & (a_arr.max(axis=(2,4,5)) > 1e-20 if a_arr.ndim > 4 else a_arr > 1e-20)
    total_paths = int(valid_mask.sum())
except Exception as e:
    try:
        # Fallback: use paths.mask if available
        mask = np.array(paths.mask) if hasattr(paths, "mask") else None
        total_paths = int(mask.sum()) if mask is not None else -1
    except:
        total_paths = -1

print(f"{len(scene.transmitters)} TX  |  {len(scene.receivers)} RX  |  {total_paths} paths")
scene.preview(paths=paths, show_devices=True)


